# TowerIQ Gold KPI Baseline Analysis

This notebook queries the first Gold KPI tables created from the tiny TowerIQ dataset.

Use it to manually inspect whether the Gold layer can answer practical telecom reliability questions.

Run the pipeline before this notebook if needed:

```bash
python3 data_generator/generate_tiny_dataset.py --output-dir data/raw/tiny
python3 -m src.jobs.run_bronze_ingestion --config configs/local.yaml --profile tiny
python3 -m src.jobs.run_quality_checks --config configs/local.yaml --profile tiny
python3 -m src.jobs.run_silver_transformations --config configs/local.yaml --profile tiny
python3 -m src.jobs.run_gold_kpis --config configs/local.yaml --profile tiny
```

## 1. Start Spark And Load Gold Tables

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.bronze_ingestion import build_storage_path
from src.utils.config import load_config
from src.utils.spark import create_spark_session

config = load_config("configs/local.yaml")
spark_config = config["spark"]
paths = config["paths"]

spark = create_spark_session(
    app_name="TowerIQ-GoldKpiNotebook",
    master=spark_config["master"],
    aqe_enabled=bool(spark_config["adaptive_query_execution"]),
    use_pyspark_package=bool(spark_config.get("use_pyspark_package", True)),
)

spark

In [ ]:
gold_tables = {
    "tower_daily_kpis": spark.read.parquet(build_storage_path(paths["gold"], "tiny", "tower_daily_kpis")),
    "region_daily_kpis": spark.read.parquet(build_storage_path(paths["gold"], "tiny", "region_daily_kpis")),
    "network_type_daily_kpis": spark.read.parquet(build_storage_path(paths["gold"], "tiny", "network_type_daily_kpis")),
    "subscriber_segment_daily_kpis": spark.read.parquet(build_storage_path(paths["gold"], "tiny", "subscriber_segment_daily_kpis")),
}

for table_name, df in gold_tables.items():
    df.createOrReplaceTempView(table_name)
    print(table_name, df.count())

## 2. Inspect Gold Schemas

In [ ]:
gold_tables["tower_daily_kpis"].printSchema()

In [ ]:
gold_tables["region_daily_kpis"].printSchema()

In [ ]:
gold_tables["network_type_daily_kpis"].printSchema()

In [ ]:
gold_tables["subscriber_segment_daily_kpis"].printSchema()

## 3. Tower-Level Reliability Questions

### Q1. Which towers have the lowest health scores?

In [ ]:
spark.sql("""
SELECT
  event_date,
  tower_id,
  region_name,
  tower_type,
  total_network_events,
  network_failure_rate,
  dropped_call_rate,
  failed_session_rate,
  critical_alarms,
  tower_health_score
FROM tower_daily_kpis
ORDER BY tower_health_score ASC
LIMIT 10
""").show(truncate=False)

### Q2. Which towers have the highest network failure rate?

In [ ]:
spark.sql("""
SELECT
  event_date,
  tower_id,
  region_name,
  tower_type,
  total_network_events,
  failed_network_events,
  network_failure_rate,
  tower_health_score
FROM tower_daily_kpis
WHERE total_network_events >= 50
ORDER BY network_failure_rate DESC
LIMIT 10
""").show(truncate=False)

### Q3. Which towers have the highest dropped-call rate?

In [ ]:
spark.sql("""
SELECT
  event_date,
  tower_id,
  region_name,
  total_calls,
  dropped_calls,
  dropped_call_rate,
  tower_health_score
FROM tower_daily_kpis
WHERE total_calls >= 10
ORDER BY dropped_call_rate DESC
LIMIT 10
""").show(truncate=False)

### Q4. Which towers carry high traffic but have weak health?

In [ ]:
spark.sql("""
SELECT
  event_date,
  tower_id,
  region_name,
  tower_type,
  total_network_events,
  total_data_mb,
  tower_health_score
FROM tower_daily_kpis
WHERE total_network_events >= 80
ORDER BY tower_health_score ASC, total_network_events DESC
LIMIT 10
""").show(truncate=False)

### Q5. Which tower types perform worse on average?

In [ ]:
spark.sql("""
SELECT
  tower_type,
  COUNT(DISTINCT tower_id) AS towers,
  ROUND(AVG(tower_health_score), 3) AS avg_health_score,
  ROUND(AVG(network_failure_rate), 3) AS avg_network_failure_rate,
  ROUND(AVG(dropped_call_rate), 3) AS avg_dropped_call_rate,
  ROUND(AVG(failed_session_rate), 3) AS avg_failed_session_rate
FROM tower_daily_kpis
GROUP BY tower_type
ORDER BY avg_health_score ASC
""").show(truncate=False)

## 4. Region-Level Reliability Questions

### Q6. Which regions have the worst average tower health?

In [ ]:
spark.sql("""
SELECT
  event_date,
  region_id,
  region_name,
  zone,
  active_towers,
  avg_tower_health_score,
  network_failure_rate,
  dropped_call_rate,
  failed_session_rate,
  critical_alarms
FROM region_daily_kpis
ORDER BY avg_tower_health_score ASC
LIMIT 10
""").show(truncate=False)

### Q7. Which regions have the highest dropped-call rate?

In [ ]:
spark.sql("""
SELECT
  event_date,
  region_name,
  total_calls,
  dropped_calls,
  dropped_call_rate,
  avg_tower_health_score
FROM region_daily_kpis
WHERE total_calls >= 50
ORDER BY dropped_call_rate DESC
LIMIT 10
""").show(truncate=False)

### Q8. Which zones perform worse overall?

In [ ]:
spark.sql("""
SELECT
  zone,
  COUNT(DISTINCT region_id) AS regions,
  ROUND(AVG(avg_tower_health_score), 3) AS avg_health_score,
  ROUND(AVG(network_failure_rate), 3) AS avg_network_failure_rate,
  ROUND(AVG(dropped_call_rate), 3) AS avg_dropped_call_rate,
  ROUND(AVG(failed_session_rate), 3) AS avg_failed_session_rate,
  SUM(critical_alarms) AS critical_alarms
FROM region_daily_kpis
GROUP BY zone
ORDER BY avg_health_score ASC
""").show(truncate=False)

## 5. Network-Type Performance Questions

### Q9. Is 5G performing better or worse than 4G/LTE?

In [ ]:
spark.sql("""
SELECT
  network_type,
  SUM(total_network_events) AS total_network_events,
  SUM(failed_network_events) AS failed_network_events,
  ROUND(AVG(network_failure_rate), 3) AS avg_network_failure_rate,
  ROUND(AVG(avg_network_latency_ms), 3) AS avg_network_latency_ms,
  ROUND(AVG(avg_signal_strength_dbm), 3) AS avg_signal_strength_dbm,
  ROUND(SUM(total_data_mb), 3) AS total_data_mb
FROM network_type_daily_kpis
GROUP BY network_type
ORDER BY avg_network_failure_rate DESC
""").show(truncate=False)

### Q10. Which network type has the highest failed-session rate by day?

In [ ]:
spark.sql("""
SELECT
  event_date,
  network_type,
  total_data_sessions,
  failed_data_sessions,
  failed_session_rate,
  ROUND(avg_session_latency_ms, 3) AS avg_session_latency_ms,
  total_data_mb
FROM network_type_daily_kpis
ORDER BY failed_session_rate DESC
LIMIT 10
""").show(truncate=False)

## 6. Subscriber Segment And Plan Questions

### Q11. Which customer segment and plan type has the worst dropped-call rate?

In [ ]:
spark.sql("""
SELECT
  event_date,
  customer_segment,
  plan_type,
  total_calls,
  dropped_calls,
  dropped_call_rate,
  failed_call_rate
FROM subscriber_segment_daily_kpis
WHERE total_calls >= 20
ORDER BY dropped_call_rate DESC
LIMIT 10
""").show(truncate=False)

### Q12. Which customer segment consumes the most data?

In [ ]:
spark.sql("""
SELECT
  customer_segment,
  plan_type,
  SUM(total_data_sessions) AS total_data_sessions,
  ROUND(SUM(total_data_mb), 3) AS total_data_mb,
  ROUND(AVG(failed_session_rate), 3) AS avg_failed_session_rate
FROM subscriber_segment_daily_kpis
GROUP BY customer_segment, plan_type
ORDER BY total_data_mb DESC
LIMIT 10
""").show(truncate=False)

### Q13. Are enterprise users experiencing reliability issues?

In [ ]:
spark.sql("""
SELECT
  event_date,
  customer_segment,
  plan_type,
  total_calls,
  dropped_call_rate,
  failed_call_rate,
  total_data_sessions,
  failed_session_rate,
  total_data_mb
FROM subscriber_segment_daily_kpis
WHERE customer_segment = 'enterprise' OR plan_type = 'enterprise'
ORDER BY event_date, customer_segment, plan_type
""").show(50, truncate=False)

## 7. Baseline Summary Queries

### Q14. One-row KPI summary for the tiny Gold dataset

In [ ]:
spark.sql("""
SELECT
  COUNT(DISTINCT tower_id) AS towers,
  COUNT(DISTINCT region_id) AS regions,
  SUM(total_network_events) AS total_network_events,
  SUM(failed_network_events) AS failed_network_events,
  ROUND(AVG(network_failure_rate), 3) AS avg_network_failure_rate,
  SUM(total_calls) AS total_calls,
  SUM(dropped_calls) AS dropped_calls,
  ROUND(AVG(dropped_call_rate), 3) AS avg_dropped_call_rate,
  SUM(total_data_sessions) AS total_data_sessions,
  SUM(failed_data_sessions) AS failed_data_sessions,
  ROUND(AVG(failed_session_rate), 3) AS avg_failed_session_rate,
  ROUND(SUM(total_data_mb), 3) AS total_data_mb,
  SUM(critical_alarms) AS critical_alarms,
  ROUND(AVG(tower_health_score), 3) AS avg_tower_health_score
FROM tower_daily_kpis
""").show(truncate=False)

### Q15. Which dimensions should we investigate first?

In [ ]:
spark.sql("""
WITH tower_risk AS (
  SELECT 'tower' AS dimension_type, tower_id AS dimension_value,
         ROUND(AVG(tower_health_score), 3) AS score
  FROM tower_daily_kpis
  GROUP BY tower_id
),
region_risk AS (
  SELECT 'region' AS dimension_type, region_name AS dimension_value,
         ROUND(AVG(avg_tower_health_score), 3) AS score
  FROM region_daily_kpis
  GROUP BY region_name
)
SELECT * FROM tower_risk
UNION ALL
SELECT * FROM region_risk
ORDER BY score ASC
LIMIT 15
""").show(truncate=False)

## 8. Stop Spark

Run this when you are done with the notebook.

In [ ]:
spark.stop()